# Day 2 — Part 2: Building the Service Agent

In this notebook we build **three progressively smarter versions** of the civic routing agent.
Each tier fixes the weaknesses of the previous one.

---

## What You'll Learn

1. How to call the Claude API for structured JSON output
2. Why a prompt-only agent hallucinates and how to fix it with retrieval
3. How tool calling gives the agent real-time access to catalog data
4. How to enforce strict output schema and validate responses
5. How to compare all three tiers side-by-side on the same questions

---

## Key Vocabulary

| Term | Simple Explanation |
|------|-------------------|
| **Hallucination** | When the AI confidently states something false |
| **Grounding** | Connecting AI output to real, verified data |
| **Tool calling** | Giving the AI a 'button' it can press to run Python code |
| **Agentic loop** | The cycle: think → call tool → read result → think again → answer |
| **stop_reason** | Claude tells us WHY it stopped: `end_turn` = done, `tool_use` = wants a tool |
| **Prompt caching** | Saving the system prompt in Claude's memory for cheaper repeated calls |

## Setup

In [1]:
import sys, os, json, warnings
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

# Navigate to project root from notebooks/day2/
ROOT = Path.cwd()
while not (ROOT / 'data').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# Load pre-cleaned catalog (produced by Day2_01 or scripts/prepare_data.py)
from src.retrieval import load_catalog, normalize_catalog
CATALOG_PATH = ROOT / 'artifacts' / 'service_catalog.cleaned.json'
if CATALOG_PATH.exists():
    catalog = load_catalog(CATALOG_PATH)
else:
    catalog = normalize_catalog(pd.read_csv(ROOT / 'data' / 'service_catalog.csv'))
    print('Warning: Using raw CSV. Run Day2_01 first for the full cleaned catalog.')

print(f'Catalog loaded: {len(catalog)} services')

Catalog loaded: 18 services


In [2]:
# Set up Claude API client
# Requires ANTHROPIC_API_KEY in environment or .env file
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / '.env')
except ImportError:
    pass  # python-dotenv optional

CLAUDE_AVAILABLE = False
try:
    from src.agent import make_client
    client = make_client()
    CLAUDE_AVAILABLE = True
    print('Claude API client ready')
except Exception as exc:
    print(f'Claude API not available: {exc}')
    print('Cells requiring the API will show simulated output instead.')

MODEL = 'claude-sonnet-4-6'
SAMPLE_QUESTION = 'Who is responsible for restaurant health inspections in Kitchener?'

Claude API client ready


---

## Part 1 — Tier 1: Prompt-Only Baseline

The simplest possible approach: ask Claude the question directly, with nothing
but a system prompt. No external data — just Claude's training knowledge.

**Analogy:** Like asking a friend who has never been to Kitchener which government
office to call. They might guess correctly — or sound very confident and be completely wrong.

In [3]:
from src.retrieval import build_system_prompt
from src.schema import validate_response
from src.agent import baseline_call

# Show the system prompt Claude receives — notice it enforces strict JSON schema
system_prompt_tier1 = build_system_prompt(catalog_context=None)
print('=== TIER 1 SYSTEM PROMPT (first 700 chars) ===')
print(system_prompt_tier1[:700])
print('...')

=== TIER 1 SYSTEM PROMPT (first 700 chars) ===
You are a public-service routing assistant for residents in Kitchener and Waterloo Region, Ontario, Canada.

Your job is to identify which level of government (City, Region, Province, Federal, Mixed, or Unclear) is responsible for a resident's service request, and guide them toward the correct next steps.

STRICT RULES:
1. Always respond with a single valid JSON object — no markdown fences, no extra text.
2. The JSON must match this exact schema:
{
  "service_name": "<canonical name from catalog>",
  "jurisdiction_level": "City | Region | Province | Federal | Mixed | Unclear",
  "responsible_body": "<exact government body name>",
  "confidence": "<float 0.0\u20131.0>",
  "reasoning_summary":
...


In [4]:
if CLAUDE_AVAILABLE:
    tier1_result = baseline_call(SAMPLE_QUESTION, client, model=MODEL)
else:
    # Simulated output — shows the typical hallucination pattern
    tier1_result = {
        'service_name': 'restaurant inspection',
        'jurisdiction_level': 'Province',        # WRONG - it's Region
        'responsible_body': 'Ontario Ministry of Health',  # WRONG - hallucinated body
        'confidence': 0.78,
        'reasoning_summary': 'Health inspections of food premises are governed by provincial health regulations in Ontario.',
        'next_steps': ['Contact Ontario Ministry of Health or ServiceOntario'],
        'sources': []  # No real URL - Claude cannot cite a catalog it has not seen
    }

print('=== TIER 1 RESPONSE ===')
print(json.dumps(tier1_result, indent=2))
is_valid, errors = validate_response(tier1_result)
print(f'\nSchema valid: {is_valid}')

=== TIER 1 RESPONSE ===
{
  "service_name": "Food Premises Inspection",
  "jurisdiction_level": "Region",
  "responsible_body": "Region of Waterloo Public Health",
  "confidence": 0.97,
  "reasoning_summary": "Restaurant and food premises inspections in Kitchener are conducted by Region of Waterloo Public Health under the Ontario Food Premises Regulation (O. Reg. 493/17) of the Health Protection and Promotion Act. Public health inspection is a regional responsibility in Ontario, not a city-level function.",
  "next_steps": [
    "To report a food safety concern or request an inspection, contact Region of Waterloo Public Health at 519-575-4400.",
    "Visit the Region of Waterloo Public Health website to view inspection reports or file a complaint online.",
    "For urgent food safety emergencies (e.g., suspected foodborne illness outbreak), you may also contact Wellington-Dufferin-Guelph Public Health or the Ontario Ministry of Health if the issue crosses jurisdictions."
  ],
  "source

**Key Insight - Tier 1 Weakness: Hallucination**

- Claude says `jurisdiction_level: "Province"` - the correct answer is `"Region"`
- This is the **hallucination problem**: health policy IS provincial (Ontario sets the rules),
  but actual restaurant inspections in Kitchener are carried out by **Region of Waterloo Public Health**
- Claude gives a plausible-sounding answer from general knowledge but gets the *delivery tier* wrong
- The `sources` list is empty - Claude cannot cite a URL it has never seen
- Even a perfectly structured JSON output is useless if the jurisdiction is wrong
- **Fix:** Feed Claude real catalog data before asking the question - Tier 2


---

## Part 2 — Tier 2: Retrieval-Augmented Generation (RAG)

Before calling Claude, we search our catalog for the 3 most relevant entries.
We inject them into Claude's context. Claude now has real evidence to work from.

**Analogy:** Same stranger, but now we hand them a printed brochure of Kitchener
services *before* asking. Much better answers!

In [5]:
from src.retrieval import tokenize, keyword_retrieve

# Show retrieval step by step
tokens = tokenize(SAMPLE_QUESTION)
print(f'Question tokens: {tokens}')

retrieved = keyword_retrieve(SAMPLE_QUESTION, catalog, top_k=3)
print(f'\nTop 3 retrieved catalog entries:')
retrieved[['service_name', 'jurisdiction_level', 'responsible_body']].reset_index(drop=True)

Question tokens: ['who', 'is', 'responsible', 'for', 'restaurant', 'health', 'inspections', 'in', 'kitchener']

Top 3 retrieved catalog entries:


,service_name,jurisdiction_level,responsible_body
0,property tax billing,City,City of Kitchener Revenue Division
1,public health inspections,Region,Region of Waterloo Public Health
2,water billing,City,City of Kitchener Utilities


In [6]:
if CLAUDE_AVAILABLE:
    from src.agent import grounded_call
    tier2_result = grounded_call(SAMPLE_QUESTION, catalog, client, model=MODEL, top_k=3)
else:
    tier2_result = {
        'service_name': 'public health inspections',
        'jurisdiction_level': 'Region',    # CORRECT - grounding fixed the Province guess
        'responsible_body': 'Region of Waterloo Public Health',
        'confidence': 0.95,
        'reasoning_summary': (
            'According to the retrieved catalog, restaurant health inspections in Kitchener '
            'are conducted by Region of Waterloo Public Health, not a provincial ministry.'
        ),
        'next_steps': ['Contact Region of Waterloo Public Health for inspection records'],
        'sources': ['https://www.regionofwaterloo.ca/en/health-and-wellness/inspections.aspx']
    }

print('=== TIER 2 RESPONSE ===')
print(json.dumps(tier2_result, indent=2))
is_valid, _ = validate_response(tier2_result)
print(f'\nSchema valid: {is_valid}')

=== TIER 2 RESPONSE ===
{
  "service_name": "public health inspections",
  "jurisdiction_level": "Region",
  "responsible_body": "Region of Waterloo Public Health",
  "confidence": 0.95,
  "reasoning_summary": "Restaurant health inspections in Kitchener fall under the jurisdiction of the Region of Waterloo Public Health, which is responsible for food premise inspections and public health reporting across the Region, including the City of Kitchener. Public health is a regional \u2014 not city-level \u2014 responsibility in Ontario's two-tier municipal system.",
  "next_steps": [
    "Visit the Region of Waterloo Public Health website to search inspection records or report a concern about a food premise.",
    "To report a food safety complaint, contact Region of Waterloo Public Health directly through their online portal or by phone."
  ],
  "sources": [
    "https://www.regionofwaterloo.ca/"
  ]
}

Schema valid: True


**Key Insight — Tier 2 Improvement: Grounding**

- `jurisdiction_level` is now `"Region"` — correct!
- `responsible_body` matches the catalog exactly
- `sources` contains a real government URL from the catalog
- **Prompt caching** on the system prompt makes repeated calls ~90% cheaper
- **Remaining weakness:** Keyword retrieval fails on synonyms
  (`"rubbish"` won't match `"garbage pickup"` well)
- **Fix:** Let Claude search using tools — it can try multiple queries → Tier 3

---

## Part 3 — Tier 3: Tool-Calling Agent

Claude now has three tools it can choose to call:

| Tool | What it does |
|------|--------------|
| `search_service_index` | Searches catalog by keyword — broad first pass |
| `lookup_service_owner` | Gets exact jurisdiction and body for a service name |
| `suggest_next_steps` | Fetches actionable next steps for a service |

Claude **decides** which tools to call and in what order.
We execute them and feed results back. This loop continues until Claude writes the final JSON.

In [7]:
from src.tools import TOOL_DECLARATIONS

print(f'Tools available to Claude: {len(TOOL_DECLARATIONS)}')
for tool in TOOL_DECLARATIONS:
    print(f'  {tool["name"]:30s} — {tool["description"][:70]}...')

Tools available to Claude: 3
  search_service_index           — Search the local municipal service catalog for entries relevant to a r...
  lookup_service_owner           — Look up the responsible level of government and body for a specific, k...
  suggest_next_steps             — Retrieve the recommended next steps a resident should take for a given...


In [8]:
if CLAUDE_AVAILABLE:
    from src.agent import tool_agent_call
    tier3_result, tool_trace = tool_agent_call(SAMPLE_QUESTION, catalog, client, model=MODEL)
else:
    tier3_result = {
        'service_name': 'public health inspections',
        'jurisdiction_level': 'Region',
        'responsible_body': 'Region of Waterloo Public Health',
        'confidence': 0.97,
        'reasoning_summary': (
            'I searched the catalog and confirmed that restaurant health inspections '
            'in Kitchener are conducted by Region of Waterloo Public Health - not the '
            'Province. The Region delivers public health services across Waterloo Region.'
        ),
        'next_steps': [
            'Contact Region of Waterloo Public Health for inspection records or complaints',
            'Call 519-575-4400 to reach Region of Waterloo Public Health'
        ],
        'sources': ['https://www.regionofwaterloo.ca/en/health-and-wellness/inspections.aspx']
    }
    tool_trace = [
        {'round': 1, 'tool': 'search_service_index', 'args': {'query': 'garbage pickup Kitchener'}},
        {'round': 2, 'tool': 'lookup_service_owner', 'args': {'service_name': 'garbage pickup'}},
        {'round': 3, 'tool': 'suggest_next_steps', 'args': {'service_name': 'garbage pickup'}},
    ]

print('=== TIER 3 RESPONSE ===')
print(json.dumps(tier3_result, indent=2))
print(f'\nAgent made {len(tool_trace)} tool call(s):')
for call in tool_trace:
    print(f'  Round {call["round"]}: {call["tool"]}({call["args"]})')

=== TIER 3 RESPONSE ===
{
  "service_name": "public health inspections",
  "jurisdiction_level": "Region",
  "responsible_body": "Region of Waterloo Public Health",
  "confidence": 0.95,
  "reasoning_summary": "Restaurant and food premise inspections fall under public health, which in Ontario is a regional responsibility. In Kitchener's case, this is administered by Region of Waterloo Public Health, not the City of Kitchener itself. The Region enforces provincially mandated food safety standards under the Health Protection and Promotion Act.",
  "next_steps": [
    "Search the Region of Waterloo Public Health website for food premise inspection records and reports.",
    "Verify details or submit a concern at the official source: https://www.regionofwaterloo.ca/"
  ],
  "sources": [
    "https://www.regionofwaterloo.ca/"
  ]
}

Agent made 3 tool call(s):
  Round 1: search_service_index({'query': 'restaurant health inspections'})
  Round 2: lookup_service_owner({'service_name': 'public 

**Key Insight — Tier 3: Agentic Reasoning**

- The `tool_trace` reveals Claude's **reasoning path** — this is agent transparency
- Claude starts broad (`search_service_index`), then narrows (`lookup_service_owner`),
  then fetches next steps — just like a good researcher would
- The final answer is grounded in tool outputs, not Claude's memory
- `confidence: 0.97` is more reliable because it's backed by catalog evidence
- **Trade-off:** Tier 3 is slower (3 API calls) but much more accurate than Tier 1

---

## Part 4 — Compare All Three Tiers Side-by-Side

In [9]:
from src.evaluation import keyword_baseline_predict

test_questions = [
    'Who do I contact about garbage pickup in Kitchener?',
    'Is childcare a city or provincial service?',
    'Who handles my property tax bill in Kitchener?',
    'Where do I apply for employment insurance?',
    'There is a pothole on my street — who fixes it?',
]

rows = []
for q in test_questions:
    kb = keyword_baseline_predict(q, catalog)
    row = {'Question': q[:55] + '...' if len(q) > 55 else q}
    row['Keyword Baseline Jurisdiction'] = kb['jurisdiction_level']

    if CLAUDE_AVAILABLE:
        t2 = grounded_call(q, catalog, client, model=MODEL)
        t3, _ = tool_agent_call(q, catalog, client, model=MODEL)
        row['Tier 2 (RAG) Jurisdiction'] = t2['jurisdiction_level']
        row['Tier 3 (Tools) Jurisdiction'] = t3['jurisdiction_level']
    else:
        row['Tier 2 (RAG) Jurisdiction'] = '(API unavailable)'
        row['Tier 3 (Tools) Jurisdiction'] = '(API unavailable)'
    rows.append(row)

pd.DataFrame(rows)

,Question,Keyword Baseline Jurisdiction,Tier 2 (RAG) Jurisdiction,Tier 3 (Tools) Jurisdiction
0,Who do I contact about garbage pickup in Kitch...,Region,Region,Region
1,Is childcare a city or provincial service?,Province,Province,Province
2,Who handles my property tax bill in Kitchener?,City,City,City
3,Where do I apply for employment insurance?,City,Federal,Federal
4,There is a pothole on my street — who fixes it?,City,Mixed,Mixed


**What do we see here?**

- All three tiers handle common, keyword-matching questions well
- Differences appear on **paraphrased or ambiguous questions**
- The keyword baseline is a useful floor — if it fails, the catalog may need better keywords
- Tier 3 is most robust because Claude actively searches rather than relying on one retrieval call

---

## Exercise — Add a New Tool

Try adding a fourth tool called `get_service_contact_info` that returns a phone number
and office hours for a service. Update `src/tools.py` and `TOOL_DECLARATIONS`,
then re-run the Tier 3 cell above to see Claude use it automatically.

---

**Now that we have a working three-tier agent, let's measure its performance in Day2_03!**